In [4]:
# =============================================================================
# 21cmFAST v3 — M_TURN & F_ESC10 Parameter Play
# Goal: vary reionisation morphology, characterise bubble sizes
#       (Asthana+2024 style), prep for kSZ from lightcones
# Date: 29 May 2026
# =============================================================================

# ── CELL 1 : Imports & directory setup ────────────────────────────────────────
import os
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy import ndimage
import py21cmfast as p21c
from py21cmfast import global_params
import matplotlib
matplotlib.use('Agg')   # non-interactive, no display needed
import matplotlib as mpl
import matplotlib.pyplot as plt

# ── Output directories ────────────────────────────────────────────────────────
RUN_DIR   = os.path.expanduser("~/29May2026_parameter_M_turn_f_esc")
PLOT_DIR  = os.path.join(RUN_DIR, "plots")
CACHE_DIR = os.path.join(RUN_DIR, "cache")
os.makedirs(PLOT_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

# Tell 21cmFAST where to write its cache
p21c.config["direc"] = CACHE_DIR

print(f"py21cmfast version : {p21c.__version__}")
print(f"Cache  → {CACHE_DIR}")
print(f"Plots  → {PLOT_DIR}")


py21cmfast version : 3.3.1
Cache  → /user1/swanith/29May2026_parameter_M_turn_f_esc/cache
Plots  → /user1/swanith/29May2026_parameter_M_turn_f_esc/plots


In [5]:
# ── CELL 2 : Plotting style (run once, never override) ───────────────────────
# Modelled on Parameter_play.py — two contexts: PDF (paper-quality)
# and PNG (screen). Use save_fig() or save_fig_multi() for all saves.

PDF_STYLE = {
    'font.family'         : 'serif',
    'font.serif'          : ['Times New Roman', 'DejaVu Serif'],
    'mathtext.fontset'    : 'cm',
    'font.size'           : 30,
    'axes.labelsize'      : 29,
    'axes.titlesize'      : 40,
    'xtick.labelsize'     : 30,
    'ytick.labelsize'     : 30,
    'legend.fontsize'     : 20,
    'figure.titlesize'    : 28,
    'xtick.direction'     : 'in',
    'ytick.direction'     : 'in',
    'xtick.top'           : True,
    'ytick.right'         : True,
    'xtick.minor.visible' : True,
    'ytick.minor.visible' : True,
    'xtick.major.size'    : 6,
    'ytick.major.size'    : 6,
    'xtick.minor.size'    : 3,
    'ytick.minor.size'    : 3,
    'xtick.major.width'   : 1.0,
    'ytick.major.width'   : 1.0,
    'xtick.minor.width'   : 0.8,
    'ytick.minor.width'   : 0.8,
    'axes.linewidth'      : 1.0,
    'lines.linewidth'     : 1.8,
    'lines.markersize'    : 5,
    'figure.dpi'          : 150,
    'savefig.dpi'         : 300,
    'savefig.bbox'        : 'tight',
    'savefig.pad_inches'  : 0.05,
}

PNG_STYLE = {
    'font.family'         : 'serif',
    'font.serif'          : ['Times New Roman', 'DejaVu Serif'],
    'mathtext.fontset'    : 'cm',
    'font.size'           : 15,
    'axes.labelsize'      : 25,
    'axes.titlesize'      : 18,
    'xtick.labelsize'     : 25,
    'ytick.labelsize'     : 25,
    'legend.fontsize'     : 18,
    'figure.titlesize'    : 15,
    'xtick.direction'     : 'in',
    'ytick.direction'     : 'in',
    'xtick.top'           : True,
    'ytick.right'         : True,
    'xtick.minor.visible' : True,
    'ytick.minor.visible' : True,
    'axes.linewidth'      : 1.0,
    'lines.linewidth'     : 1.5,
    'figure.dpi'          : 150,
    'savefig.dpi'         : 300,
    'savefig.bbox'        : 'tight',
    'savefig.pad_inches'  : 0.05,
}


def save_fig(plot_func, name, title=None, figsize=(10, 7)):
    """
    Single-axes save: calls plot_func(ax) twice — once for PDF, once for PNG.
    PDF gets no title; PNG gets `title` as a bold suptitle.
    """
    with mpl.rc_context(PDF_STYLE):
        fig, ax = plt.subplots(figsize=figsize, constrained_layout=True)
        plot_func(ax)
        ax.set_title("")
        fig.savefig(os.path.join(PLOT_DIR, f"{name}.pdf"))
        plt.close(fig)

    with mpl.rc_context(PNG_STYLE):
        fig, ax = plt.subplots(figsize=figsize, constrained_layout=True)
        plot_func(ax)
        if title:
            ax.set_title(title, fontweight='bold')
        fig.savefig(os.path.join(PLOT_DIR, f"{name}.png"))
        plt.close(fig)
    print(f"  Saved: {name}.{{pdf,png}}")


def save_fig_multi(plot_func, name, title=None, figsize=(20, 4.2)):
    """
    Multi-panel save: calls plot_func(fig) which creates its own axes.
    """
    with mpl.rc_context(PDF_STYLE):
        fig = plt.figure(figsize=figsize, constrained_layout=True)
        plot_func(fig)
        fig.savefig(os.path.join(PLOT_DIR, f"{name}.pdf"))
        plt.close(fig)

    with mpl.rc_context(PNG_STYLE):
        fig = plt.figure(figsize=figsize, constrained_layout=True)
        plot_func(fig)
        if title:
            fig.suptitle(title, fontweight='bold')
        fig.savefig(os.path.join(PLOT_DIR, f"{name}.png"))
        plt.close(fig)
    print(f"  Saved: {name}.{{pdf,png}}")


print("✓ Plotting style loaded")

✓ Plotting style loaded


In [11]:
# ── CELL 3 : Simulation parameters ───────────────────────────────────────────
# UserParams: box size. Keep small for exploration; scale up for production.
# 128^3 at 200 Mpc is fast but resolves >~1.5 Mpc bubbles adequately.
# For lightcone production runs: HII_DIM=256, BOX_LEN=300 recommended.

USER_PARAMS = p21c.UserParams(
    HII_DIM   = 256,
    DIM       = 512,    # 2× HII_DIM is sufficient for coeval
    BOX_LEN   = 400.0,
    N_THREADS = 8,
)

COSMO_PARAMS = p21c.CosmoParams(
    SIGMA_8  = 0.829,
    hlittle  = 0.678,
    OMb      = 0.0482,
    OMm      = 0.308,
    POWER_INDEX = 0.961,
)

# FlagOptions: USE_MASS_DEPENDENT_ZETA=True activates M_TURN and F_ESC10/F_STAR10.
# Without it, only HII_EFF_FACTOR matters.
FLAG_OPTIONS = p21c.FlagOptions(
    USE_MASS_DEPENDENT_ZETA = True,
    USE_TS_FLUCT            = False,   # spin-temperature fluctuations
    INHOMO_RECO             =False,    # inhomogeneous recombinations
    USE_MINI_HALOS          = False,  # keep False unless you want the mini-halo path
)

# Redshift range for lightcones
Z_MIN    = 5.5
Z_MAX    = 12.0
RANDOM_SEED = 12345

print("UserParams:", USER_PARAMS)
print("FlagOptions:", FLAG_OPTIONS)
print("\nCosmoParams:", COSMO_PARAMS)
print("\nAstroParams (default):", p21c.AstroParams())



UserParams: UserParams:
    BOX_LEN                 : 400.0
    DIM                     : 512
    FAST_FCOLL_TABLES       : False
    HII_DIM                 : 256
    HMF                     : 1
    MINIMIZE_MEMORY         : False
    NON_CUBIC_FACTOR        : 1.0
    NO_RNG                  : False
    N_THREADS               : 8
    PERTURB_ON_HIGH_RES     : False
    POWER_SPECTRUM          : 0
    USE_2LPT                : True
    USE_FFTW_WISDOM         : False
    USE_INTERPOLATION_TABLES: True
    USE_RELATIVE_VELOCITIES : False
    
FlagOptions: FlagOptions:
    FIX_VCB_AVG            : False
    INHOMO_RECO            : False
    M_MIN_in_Mass          : True
    PHOTON_CONS            : False
    SUBCELL_RSD            : False
    USE_CMB_HEATING        : True
    USE_HALO_FIELD         : False
    USE_LYA_HEATING        : True
    USE_MASS_DEPENDENT_ZETA: True
    USE_MINI_HALOS         : False
    USE_TS_FLUCT           : False
    

CosmoParams: CosmoParams:
    OMb     

/user1/swanith/.conda/envs/p21c_v3/lib/python3.10/site-packages/py21cmfast/inputs.py:491: UserWarning: The USE_INTERPOLATION_TABLES setting has changed in v3.1.2 to be default True. You can likely ignore this warning, but if you relied onhaving USE_INTERPOLATION_TABLES=False by *default*, please set it explicitly. To silence this warning, set it explicitly to True. Thiswarning will be removed in v4.
  warnings.warn(


In [21]:
# ── CELL 4 : Parameter grid ───────────────────────────────────────────────────
import itertools

M_TURN_VALUES  = [7.5, 8.0, 8.7, 9.2, 9.8]
F_ESC10_VALUES = [-2.0, -1.5, -1.0, -0.5, -0.2]

# Fixed for all models
ALPHA_ESC_FIX  = -0.5
F_STAR10_FIX   = -1.3
RANDOM_SEED    = 12345

# Colour maps: M_TURN → hue family, F_ESC10 → shade within family
import matplotlib.cm as cm
cmap_mt = cm.get_cmap('tab10', len(M_TURN_VALUES))

SNAP_REDSHIFTS = [
    20.0, 18.0, 16.0, 14.0, 12.0,
    10.0,  9.0,  8.0,  7.5,  7.0,
     6.5,  6.0,  5.5,  5.0,
]

def make_label(mt, fe):
    fe_str = f"{fe:.1f}".replace('-', 'm')
    return f"MT{mt:.1f}_FE{fe_str}"

# Build PARAM_SETS as ordered dict
PARAM_SETS = {}
for i, mt in enumerate(M_TURN_VALUES):
    for j, fe in enumerate(F_ESC10_VALUES):
        label = make_label(mt, fe)
        # colour: hue from M_TURN, alpha/shade from F_ESC10
        base_col = list(cmap_mt(i))
        base_col[3] = 0.4 + 0.6 * (j / (len(F_ESC10_VALUES) - 1))  # alpha
        PARAM_SETS[label] = dict(
            M_TURN    = mt,
            F_ESC10   = fe,
            ALPHA_ESC = ALPHA_ESC_FIX,
            F_STAR10  = F_STAR10_FIX,
            color     = tuple(base_col),
            mt_idx    = i,
            fe_idx    = j,
        )

print(f"Total models : {len(PARAM_SETS)}")
print(f"Redshifts    : {len(SNAP_REDSHIFTS)}  →  {SNAP_REDSHIFTS}")
print(f"Total boxes  : {len(PARAM_SETS) * len(SNAP_REDSHIFTS)}")
print()

# ── Print grid ────────────────────────────────────────────────────────────────
print(f"{'':>12}", end='')
for fe in F_ESC10_VALUES:
    print(f"  FE={fe:+.1f}", end='')
print()
print("-" * (12 + 10 * len(F_ESC10_VALUES)))
for mt in M_TURN_VALUES:
    print(f"MT={mt:.1f}      ", end='')
    for fe in F_ESC10_VALUES:
        print(f"  {make_label(mt,fe)[:12]}", end='')
    print()

Total models : 25
Redshifts    : 14  →  [20.0, 18.0, 16.0, 14.0, 12.0, 10.0, 9.0, 8.0, 7.5, 7.0, 6.5, 6.0, 5.5, 5.0]
Total boxes  : 350

              FE=-2.0  FE=-1.5  FE=-1.0  FE=-0.5  FE=-0.2
--------------------------------------------------------------
MT=7.5        MT7.5_FEm2.0  MT7.5_FEm1.5  MT7.5_FEm1.0  MT7.5_FEm0.5  MT7.5_FEm0.2
MT=8.0        MT8.0_FEm2.0  MT8.0_FEm1.5  MT8.0_FEm1.0  MT8.0_FEm0.5  MT8.0_FEm0.2
MT=8.7        MT8.7_FEm2.0  MT8.7_FEm1.5  MT8.7_FEm1.0  MT8.7_FEm0.5  MT8.7_FEm0.2
MT=9.2        MT9.2_FEm2.0  MT9.2_FEm1.5  MT9.2_FEm1.0  MT9.2_FEm0.5  MT9.2_FEm0.2
MT=9.8        MT9.8_FEm2.0  MT9.8_FEm1.5  MT9.8_FEm1.0  MT9.8_FEm0.5  MT9.8_FEm0.2


In [22]:
# ── CELL 5 : Run coeval boxes (or load from cache) ────────────────────────────
import pickle, time

def coeval_cache_path(label, z):
    return os.path.join(CACHE_DIR, f"coeval_{label}_z{z:.1f}.pkl")

def save_coeval_cache(label, z, data):
    with open(coeval_cache_path(label, z), 'wb') as f:
        pickle.dump(data, f)

def load_coeval_cache(label, z):
    p = coeval_cache_path(label, z)
    if os.path.exists(p):
        with open(p, 'rb') as f:
            return pickle.load(f)
    return None

# ── Main loop ─────────────────────────────────────────────────────────────────
coeval_data = {label: {} for label in PARAM_SETS}

total    = len(PARAM_SETS) * len(SNAP_REDSHIFTS)
done     = 0
t_start  = time.time()

for label, p in PARAM_SETS.items():
    astro = p21c.AstroParams(
        M_TURN    = p['M_TURN'],
        F_ESC10   = p['F_ESC10'],
        ALPHA_ESC = p['ALPHA_ESC'],
        F_STAR10  = p['F_STAR10'],
    )

    for z in SNAP_REDSHIFTS:
        cached = load_coeval_cache(label, z)
        if cached is not None:
            coeval_data[label][z] = cached
            done += 1
            continue

        t0 = time.time()
        coeval = p21c.run_coeval(
            redshift     = float(z),
            user_params  = USER_PARAMS,
            cosmo_params = COSMO_PARAMS,
            astro_params = astro,
            flag_options = FLAG_OPTIONS,
            random_seed  = RANDOM_SEED,
            write        = False,
        )

        data = {
            'xH_box'  : coeval.xH_box.copy(),
            'density' : coeval.density.copy(),
            'xH_mean' : float(coeval.xH_box.mean()),
        }
        del coeval

        save_coeval_cache(label, z, data)
        coeval_data[label][z] = data
        done += 1

        elapsed  = time.time() - t_start
        per_box  = elapsed / done
        remaining = (total - done) * per_box / 3600

        print(f"  [{done:>3}/{total}]  {label}  z={z:.1f}  "
              f"<xHI>={data['xH_mean']:.3f}  "
              f"t={time.time()-t0:.0f}s  "
              f"ETA={remaining:.1f}h")

print(f"\n✓ All {total} boxes ready")

# ── xHI summary grid ──────────────────────────────────────────────────────────
print(f"\n<xHI> at z=7.0:")
print(f"{'':>8}", end='')
for fe in F_ESC10_VALUES:
    print(f"  FE={fe:+.1f}", end='')
print()
print("-" * (8 + 10 * len(F_ESC10_VALUES)))
for mt in M_TURN_VALUES:
    print(f"MT={mt:.1f}  ", end='')
    for fe in F_ESC10_VALUES:
        label = make_label(mt, fe)
        xhi   = coeval_data[label].get(7.0, {}).get('xH_mean', np.nan)
        print(f"  {xhi:.3f}  ", end='')
    print()

/user1/swanith/.conda/envs/p21c_v3/lib/python3.10/site-packages/py21cmfast/_utils.py:400: UserWarning: The following parameters to FlagOptions are not supported: ['USE_VELS_AUX']
  warnings.warn(


  [  1/350]  MT7.5_FEm2.0  z=20.0  <xHI>=1.000  t=51s  ETA=4.9h
  [  2/350]  MT7.5_FEm2.0  z=18.0  <xHI>=0.999  t=51s  ETA=4.9h
  [  3/350]  MT7.5_FEm2.0  z=16.0  <xHI>=0.996  t=51s  ETA=4.9h
  [  4/350]  MT7.5_FEm2.0  z=14.0  <xHI>=0.989  t=51s  ETA=4.9h
  [  5/350]  MT7.5_FEm2.0  z=12.0  <xHI>=0.972  t=52s  ETA=4.9h
  [  6/350]  MT7.5_FEm2.0  z=10.0  <xHI>=0.932  t=51s  ETA=4.9h
  [  7/350]  MT7.5_FEm2.0  z=9.0  <xHI>=0.898  t=50s  ETA=4.9h
  [  8/350]  MT7.5_FEm2.0  z=8.0  <xHI>=0.851  t=50s  ETA=4.8h
  [  9/350]  MT7.5_FEm2.0  z=7.5  <xHI>=0.821  t=51s  ETA=4.8h
  [ 10/350]  MT7.5_FEm2.0  z=7.0  <xHI>=0.786  t=51s  ETA=4.8h
  [ 11/350]  MT7.5_FEm2.0  z=6.5  <xHI>=0.747  t=51s  ETA=4.8h
  [ 12/350]  MT7.5_FEm2.0  z=6.0  <xHI>=0.701  t=51s  ETA=4.8h
  [ 13/350]  MT7.5_FEm2.0  z=5.5  <xHI>=0.649  t=51s  ETA=4.8h
  [ 14/350]  MT7.5_FEm2.0  z=5.0  <xHI>=0.589  t=51s  ETA=4.7h
  [ 15/350]  MT7.5_FEm1.5  z=20.0  <xHI>=0.999  t=51s  ETA=4.7h
  [ 16/350]  MT7.5_FEm1.5  z=18.0  <xHI>=0.997  

In [23]:
# ── CELL 6 : Reionisation history ─────────────────────────────────────────────
import matplotlib.cm as cm

z_arr      = np.array(sorted(SNAP_REDSHIFTS))
cmap_fe    = cm.get_cmap('cool', len(F_ESC10_VALUES))   # F_ESC10 → colour
fe_colors  = {fe: cmap_fe(j) for j, fe in enumerate(F_ESC10_VALUES)}

# ── Plot: one panel per M_TURN, curves = F_ESC10 ─────────────────────────────
def plot_reion_history(fig):
    axes = fig.subplots(1, len(M_TURN_VALUES), sharey=True)

    for ax, mt in zip(axes, M_TURN_VALUES):
        for fe in F_ESC10_VALUES:
            label  = make_label(mt, fe)
            xH_arr = np.array([coeval_data[label][z]['xH_mean']
                                for z in z_arr])
            ax.plot(z_arr, xH_arr, color=fe_colors[fe],
                    lw=1.8, marker='o', ms=3,
                    label=f'FE={fe:+.1f}')

        ax.set_xlabel(r'$z$')
        ax.set_title(rf'$M_{{\rm turn}}=10^{{{mt:.1f}}}\ M_\odot$',
                     fontsize=10)
        ax.axhline(0.5, ls='--', color='gray', lw=0.8, alpha=0.6)
        ax.set_xlim(z_arr.min()-0.3, z_arr.max()+0.3)
        ax.set_ylim(-0.02, 1.05)
        ax.invert_xaxis()
        if ax == axes[0]:
            ax.set_ylabel(r'$\langle x_{\rm HI} \rangle$')

    # shared F_ESC10 legend on last panel
    axes[-1].legend(fontsize=8, loc='upper right', title=r'$\log f_{\rm esc,10}$')

    # shared colorbar
    sm = plt.cm.ScalarMappable(cmap='cool',
                                norm=plt.Normalize(min(F_ESC10_VALUES),
                                                   max(F_ESC10_VALUES)))
    sm.set_array([])
    fig.colorbar(sm, ax=axes[-1], label=r'$\log_{10} f_{\rm esc,10}$',
                 shrink=0.8)

save_fig_multi(plot_reion_history, "reion_history",
               figsize=(4 * len(M_TURN_VALUES), 5))

# ── Diagnostic PNG ────────────────────────────────────────────────────────────
with mpl.rc_context(PNG_STYLE):
    fig = plt.figure(figsize=(4 * len(M_TURN_VALUES), 5),
                     constrained_layout=True)
    plot_reion_history(fig)
    fig.suptitle('Reionisation histories — M_TURN × F_ESC10 grid',
                 fontweight='bold')
    fig.savefig(os.path.join(PLOT_DIR, "diag_reion_history.png"),
                bbox_inches='tight', dpi=150)
    plt.close(fig)
    print("  Saved: diag_reion_history.png")

# ── xHI grid table at z=7 and z=8 ────────────────────────────────────────────
for z_show in [7.0, 8.0]:
    print(f"\n<xHI> at z={z_show}:")
    print(f"{'':>8}", end='')
    for fe in F_ESC10_VALUES:
        print(f"  FE={fe:+.1f}", end='')
    print()
    print("-" * (8 + 10 * len(F_ESC10_VALUES)))
    for mt in M_TURN_VALUES:
        print(f"MT={mt:.1f}  ", end='')
        for fe in F_ESC10_VALUES:
            label = make_label(mt, fe)
            xhi   = coeval_data[label].get(z_show, {}).get('xH_mean', np.nan)
            print(f"  {xhi:.3f}  ", end='')
        print()

  Saved: reion_history.{pdf,png}
  Saved: diag_reion_history.png

<xHI> at z=7.0:
          FE=-2.0  FE=-1.5  FE=-1.0  FE=-0.5  FE=-0.2
----------------------------------------------------------
MT=7.5    0.786    0.376    0.000    0.000    0.000  
MT=8.0    0.841    0.549    0.009    0.000    0.000  
MT=8.7    0.908    0.751    0.210    0.000    0.000  
MT=9.2    0.944    0.855    0.520    0.021    0.000  
MT=9.8    0.973    0.934    0.801    0.326    0.063  

<xHI> at z=8.0:
          FE=-2.0  FE=-1.5  FE=-1.0  FE=-0.5  FE=-0.2
----------------------------------------------------------
MT=7.5    0.851    0.576    0.027    0.000    0.000  
MT=8.0    0.895    0.709    0.144    0.000    0.000  
MT=8.7    0.944    0.853    0.531    0.032    0.003  
MT=9.2    0.969    0.921    0.763    0.238    0.049  
MT=9.8    0.987    0.968    0.913    0.689    0.384  


In [25]:
# ── CELL 7 : Slice gallery + CCL bubble visualisation ────────────────────────
from scipy import ndimage
import matplotlib.colors as mcolors

def get_ccl_labelled(xH_field, xHI_thresh=0.5):
    """Return colour-labelled bubble image for visualisation."""
    ionised          = xH_field < xHI_thresh
    struct           = ndimage.generate_binary_structure(3, 1)
    labelled, n_bub  = ndimage.label(ionised, structure=struct)
    return labelled, n_bub

def make_bubble_cmap(n):
    """Random colormap with black=neutral, distinct colours per bubble."""
    rng    = np.random.default_rng(42)
    colors = [(0, 0, 0, 1)]   # label 0 = neutral = black
    colors += [(*rng.random(3), 1.0) for _ in range(max(n, 1))]
    return mcolors.ListedColormap(colors)

# ── One figure per M_TURN ─────────────────────────────────────────────────────
Z_SHOW = [10.0, 9.0, 8.0, 7.5, 7.0, 6.5, 6.0, 5.5, 5.0]  # focus redshifts
N_Z    = len(Z_SHOW)

for mt in M_TURN_VALUES:
    n_fe   = len(F_ESC10_VALUES)
    n_rows  = n_fe * 2   # xHI row + CCL row per F_ESC value
    fig, axes = plt.subplots(
        n_rows, N_Z,
        figsize=(2.5 * N_Z, 2.8 * n_rows),
        gridspec_kw=dict(wspace=0.03, hspace=0.06)
    )

    for j_fe, fe in enumerate(F_ESC10_VALUES):
        label    = make_label(mt, fe)
        row_xhi  = j_fe * 2       # xHI slice row
        row_ccl  = j_fe * 2 + 1   # CCL bubble row

        for j_z, z in enumerate(Z_SHOW):
            ax_xhi = axes[row_xhi, j_z]
            ax_ccl = axes[row_ccl, j_z]

            box = coeval_data[label][z]['xH_box']
            sl  = box[:, box.shape[1]//2, :]

            # ── xHI slice ────────────────────────────────────────
            ax_xhi.imshow(sl, origin='lower', cmap='RdBu_r',
                          vmin=0, vmax=1,
                          extent=[0, USER_PARAMS.BOX_LEN]*2)
            ax_xhi.set_xticks([]); ax_xhi.set_yticks([])

            # ── CCL bubble slice ──────────────────────────────────
            labelled, n_bub = get_ccl_labelled(box)
            lab_sl          = labelled[:, box.shape[1]//2, :]
            bcmap           = make_bubble_cmap(n_bub)
            ax_ccl.imshow(lab_sl, origin='lower',
                          cmap=bcmap,
                          vmin=0, vmax=max(n_bub, 1),
                          extent=[0, USER_PARAMS.BOX_LEN]*2)
            ax_ccl.set_xticks([]); ax_ccl.set_yticks([])
            ax_ccl.text(0.02, 0.97,
                        f'N={n_bub}',
                        transform=ax_ccl.transAxes,
                        fontsize=7, color='white', va='top')

            # ── Column titles (top row only) ──────────────────────
            if j_fe == 0:
                xhi_mean = coeval_data[label][z]['xH_mean']
                ax_xhi.set_title(rf'$z={z:.1f}$'
                                 f'\n'
                                 rf'$\langle x_{{\rm HI}}\rangle={xhi_mean:.2f}$',
                                 fontsize=8)

            # ── Row labels (first column only) ───────────────────
            if j_z == 0:
                ax_xhi.set_ylabel(rf'FE={fe:+.1f}' '\n' r'$x_{\rm HI}$',
                                  fontsize=8, rotation=90,
                                  va='center', labelpad=2)
                ax_ccl.set_ylabel(rf'FE={fe:+.1f}' '\n' 'bubbles',
                                  fontsize=8, rotation=90,
                                  va='center', labelpad=2)

    fig.suptitle(rf'$M_{{\rm turn}} = 10^{{{mt:.1f}}}\ M_\odot$ — '
                 r'$x_{\rm HI}$ slices (top) + CCL bubbles (bottom)',
                 fontsize=11, fontweight='bold', y=1.005)

    fname = f"slice_ccl_MT{mt:.1f}.png"
    fig.savefig(os.path.join(PLOT_DIR, fname),
                bbox_inches='tight', dpi=120)
    plt.close(fig)
    print(f"  Saved: {fname}")

print("\n✓ All slice+CCL figures saved")

# ── Diagnostic: single box CCL demo ──────────────────────────────────────────
# Show xHI → binary → CCL for one instructive case
with mpl.rc_context(PNG_STYLE):
    demo_label = make_label(8.7, -1.0)
    demo_z     = 8.0
    box        = coeval_data[demo_label][demo_z]['xH_box']
    sl_xhi     = box[:, box.shape[1]//2, :]
    ionised    = box < 0.5
    sl_ion     = ionised[:, box.shape[1]//2, :]
    labelled, n_bub = get_ccl_labelled(box)
    sl_lab     = labelled[:, box.shape[1]//2, :]
    bcmap      = make_bubble_cmap(n_bub)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # Panel 1: xHI
    im0 = axes[0].imshow(sl_xhi, origin='lower', cmap='RdBu_r',
                          vmin=0, vmax=1)
    axes[0].set_title(r'$x_{\rm HI}$ field', fontweight='bold')
    fig.colorbar(im0, ax=axes[0], shrink=0.8, label=r'$x_{\rm HI}$')

    # Panel 2: binary ionised mask
    axes[1].imshow(sl_ion.astype(float), origin='lower',
                   cmap='gray_r', vmin=0, vmax=1)
    axes[1].set_title(r'Ionised mask ($x_{\rm HI} < 0.5$)',
                      fontweight='bold')
    axes[1].text(0.02, 0.97, 'white = ionised\nblack = neutral',
                 transform=axes[1].transAxes, fontsize=9,
                 color='gray', va='top')

    # Panel 3: CCL labelled
    im2 = axes[2].imshow(sl_lab, origin='lower', cmap=bcmap,
                          vmin=0, vmax=max(n_bub, 1))
    axes[2].set_title(f'CCL bubbles (N={n_bub})', fontweight='bold')

    for ax in axes:
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_xlabel(f'{USER_PARAMS.BOX_LEN} cMpc')

    fig.suptitle(rf'CCL demo: {demo_label} at $z={demo_z}$  '
                 rf'$\langle x_{{\rm HI}}\rangle='
                 rf'{coeval_data[demo_label][demo_z]["xH_mean"]:.2f}$',
                 fontweight='bold')
    fig.tight_layout()
    fig.savefig(os.path.join(PLOT_DIR, "diag_ccl_demo.png"),
                bbox_inches='tight', dpi=150)
    plt.close(fig)
    print("  Saved: diag_ccl_demo.png (CCL algorithm illustration)")

  Saved: slice_ccl_MT7.5.png
  Saved: slice_ccl_MT8.0.png
  Saved: slice_ccl_MT8.7.png
  Saved: slice_ccl_MT9.2.png
  Saved: slice_ccl_MT9.8.png

✓ All slice+CCL figures saved
  Saved: diag_ccl_demo.png (CCL algorithm illustration)


In [26]:
# ── CELL 8 : Bubble size analysis ────────────────────────────────────────────

def compute_bubble_sizes(xH_field, cell_size_mpc, xHI_thresh=0.5):
    ionised          = xH_field < xHI_thresh
    struct           = ndimage.generate_binary_structure(3, 1)
    labelled, n_bub  = ndimage.label(ionised, structure=struct)

    if n_bub == 0:
        return np.array([]), cell_size_mpc**3

    parent = list(range(n_bub + 1))

    def find(x):
        while parent[x] != x:
            x = parent[x]
        return x

    for axis in range(3):
        sl1 = [slice(None)] * 3;  sl1[axis] = slice(-1, None)
        sl2 = [slice(None)] * 3;  sl2[axis] = slice(0, 1)
        edge1 = labelled[tuple(sl1)]
        edge2 = labelled[tuple(sl2)]
        mask1 = ionised[tuple(sl1)]
        mask2 = ionised[tuple(sl2)]
        for a, b in zip(edge1[mask1].ravel(), edge2[mask2].ravel()):
            ra, rb = find(a), find(b)
            if ra != rb:
                parent[ra] = rb

    remap    = np.array([find(i) for i in range(n_bub + 1)])
    labelled = remap[labelled]

    unique, counts = np.unique(labelled[labelled > 0], return_counts=True)
    V_cell  = cell_size_mpc**3
    volumes = counts * V_cell
    R_list  = (3 * volumes / (4 * np.pi))**(1/3)
    return R_list, V_cell


def cell_size_pmpc(z, box_len_cmpc, hii_dim):
    return (box_len_cmpc / hii_dim) / (1 + z)


# ── Compute ───────────────────────────────────────────────────────────────────
bubble_data = {}

for label in PARAM_SETS:
    bubble_data[label] = {}
    for z in SNAP_REDSHIFTS:
        box3d    = coeval_data[label][z]['xH_box']
        cs       = cell_size_pmpc(z, USER_PARAMS.BOX_LEN, USER_PARAMS.HII_DIM)
        R_arr, _ = compute_bubble_sizes(box3d, cs)
        bubble_data[label][z] = R_arr

        if len(R_arr) > 0:
            print(f"  {label:<22} z={z:.1f}  "
                  f"<xHI>={coeval_data[label][z]['xH_mean']:.3f}  "
                  f"N={len(R_arr):>5}  "
                  f"R_max={R_arr.max():.2f}  "
                  f"R_med={np.median(R_arr):.2f} pMpc")
        else:
            print(f"  {label:<22} z={z:.1f}  no ionised cells")

print("\n✓ Bubble analysis done")

# ── Diagnostic: N_bubbles and R_max, one panel per M_TURN ────────────────────
with mpl.rc_context(PNG_STYLE):
    fig, axes = plt.subplots(2, len(M_TURN_VALUES),
                             figsize=(4 * len(M_TURN_VALUES), 8),
                             sharey='row')

    z_arr  = sorted(SNAP_REDSHIFTS)

    for j, mt in enumerate(M_TURN_VALUES):
        ax_n = axes[0, j]
        ax_r = axes[1, j]

        for fe in F_ESC10_VALUES:
            label = make_label(mt, fe)
            col   = fe_colors[fe]

            n_bub = [len(bubble_data[label].get(z, [])) for z in z_arr]
            r_max = [bubble_data[label][z].max()
                     if len(bubble_data[label].get(z, [])) > 0
                     else 0 for z in z_arr]

            ax_n.plot(z_arr, n_bub, color=col, lw=1.5, marker='o', ms=3,
                      label=f'FE={fe:+.1f}')
            ax_r.plot(z_arr, r_max, color=col, lw=1.5, marker='o', ms=3)

        ax_n.set_title(rf'$M_{{\rm turn}}=10^{{{mt:.1f}}}$', fontsize=10)
        ax_n.invert_xaxis()
        ax_r.invert_xaxis()
        ax_r.set_xlabel(r'Redshift $z$')

        if j == 0:
            ax_n.set_ylabel('N bubbles')
            ax_r.set_ylabel(r'$R_{\rm max}$ [pMpc]')

        if j == len(M_TURN_VALUES) - 1:
            ax_n.legend(fontsize=8, title=r'$\log f_{\rm esc}$',
                        loc='upper left')

    fig.suptitle('Bubble statistics across parameter grid',
                 fontweight='bold')
    fig.tight_layout()
    fig.savefig(os.path.join(PLOT_DIR, "diag_bubble_stats.png"),
                bbox_inches='tight', dpi=150)
    plt.close(fig)
    print("  Saved: diag_bubble_stats.png")

  MT7.5_FEm2.0           z=20.0  <xHI>=1.000  N=   87  R_max=0.08  R_med=0.05 pMpc
  MT7.5_FEm2.0           z=18.0  <xHI>=0.999  N=  453  R_max=0.12  R_med=0.05 pMpc
  MT7.5_FEm2.0           z=16.0  <xHI>=0.996  N= 2008  R_max=0.18  R_med=0.06 pMpc
  MT7.5_FEm2.0           z=14.0  <xHI>=0.989  N= 7220  R_max=0.31  R_med=0.06 pMpc
  MT7.5_FEm2.0           z=12.0  <xHI>=0.972  N=20521  R_max=0.72  R_med=0.09 pMpc
  MT7.5_FEm2.0           z=10.0  <xHI>=0.932  N=44257  R_max=1.70  R_med=0.11 pMpc
  MT7.5_FEm2.0           z=9.0  <xHI>=0.898  N=56933  R_max=4.17  R_med=0.12 pMpc
  MT7.5_FEm2.0           z=8.0  <xHI>=0.851  N=65829  R_max=6.89  R_med=0.14 pMpc
  MT7.5_FEm2.0           z=7.5  <xHI>=0.821  N=67432  R_max=9.51  R_med=0.14 pMpc
  MT7.5_FEm2.0           z=7.0  <xHI>=0.786  N=66778  R_max=12.56  R_med=0.15 pMpc
  MT7.5_FEm2.0           z=6.5  <xHI>=0.747  N=64076  R_max=15.36  R_med=0.16 pMpc
  MT7.5_FEm2.0           z=6.0  <xHI>=0.701  N=59228  R_max=18.40  R_med=0.14 pMpc
  MT7.5

In [27]:
# ── CELL 9 : Volume fraction in bubbles > R_min vs redshift ──────────────────
R_MIN_LIST = [0.5, 1.0, 3.0]   # pMpc
z_sorted   = sorted(SNAP_REDSHIFTS)

def compute_vfrac(label, R_min):
    vfrac = []
    for z in z_sorted:
        R_arr = bubble_data[label].get(z, np.array([]))
        V_box = (USER_PARAMS.BOX_LEN / (1 + z))**3
        if len(R_arr) == 0:
            vfrac.append(0.0)
        else:
            V_large = np.sum((4/3) * np.pi * R_arr[R_arr >= R_min]**3)
            vfrac.append(min(V_large / V_box, 1.0))
    return np.array(vfrac)

# ── One figure per R_min ──────────────────────────────────────────────────────
for R_min in R_MIN_LIST:
    fig, axes = plt.subplots(1, len(M_TURN_VALUES),
                             figsize=(4 * len(M_TURN_VALUES), 5),
                             sharey=True)

    for ax, mt in zip(axes, M_TURN_VALUES):
        for fe in F_ESC10_VALUES:
            label  = make_label(mt, fe)
            vfrac  = compute_vfrac(label, R_min)
            ax.plot(z_sorted, vfrac, color=fe_colors[fe],
                    lw=1.8, marker='o', ms=3,
                    label=f'FE={fe:+.1f}')

        ax.set_xlabel(r'Redshift $z$')
        ax.set_title(rf'$M_{{\rm turn}}=10^{{{mt:.1f}}}$', fontsize=10)
        ax.set_xlim(min(z_sorted) - 0.3, max(z_sorted) + 0.3)
        ax.set_ylim(-0.02, 1.05)
        ax.invert_xaxis()
        ax.axhline(0.5, ls='--', color='gray', lw=0.8, alpha=0.5)

        if ax == axes[0]:
            ax.set_ylabel(r'$V_{\rm frac}(R > R_{\min})$')
        if ax == axes[-1]:
            ax.legend(fontsize=8, title=r'$\log f_{\rm esc}$',
                      loc='upper right')

    # shared colorbar
    sm = plt.cm.ScalarMappable(cmap='cool',
                                norm=plt.Normalize(min(F_ESC10_VALUES),
                                                   max(F_ESC10_VALUES)))
    sm.set_array([])
    fig.colorbar(sm, ax=axes[-1],
                 label=r'$\log_{10} f_{\rm esc,10}$', shrink=0.8)

    Rstr  = str(R_min).replace('.', 'p')
    fname = f"vfrac_Rmin{Rstr}.png"
    fig.suptitle(rf'$V_{{\rm frac}}(R > {R_min}\ \rm pMpc)$',
                 fontweight='bold')
    fig.tight_layout()
    fig.savefig(os.path.join(PLOT_DIR, fname),
                bbox_inches='tight', dpi=150)
    plt.close(fig)
    print(f"  Saved: {fname}")

# ── PDF version via save_fig_multi ───────────────────────────────────────────
def plot_vfrac_Rmin1(fig):
    axes = fig.subplots(1, len(M_TURN_VALUES), sharey=True)
    for ax, mt in zip(axes, M_TURN_VALUES):
        for fe in F_ESC10_VALUES:
            label = make_label(mt, fe)
            vfrac = compute_vfrac(label, 1.0)
            ax.plot(z_sorted, vfrac, color=fe_colors[fe],
                    lw=1.8, marker='o', ms=3, label=f'FE={fe:+.1f}')
        ax.set_xlabel(r'Redshift $z$')
        ax.set_title(rf'$M_{{\rm turn}}=10^{{{mt:.1f}}}$', fontsize=10)
        ax.set_xlim(min(z_sorted)-0.3, max(z_sorted)+0.3)
        ax.set_ylim(-0.02, 1.05)
        ax.invert_xaxis()
        ax.axhline(0.5, ls='--', color='gray', lw=0.8, alpha=0.5)
        if ax == axes[0]:
            ax.set_ylabel(r'$V_{\rm frac}(R > 1\ \rm pMpc)$')
        if ax == axes[-1]:
            ax.legend(fontsize=8, title=r'$\log f_{\rm esc}$')

save_fig_multi(plot_vfrac_Rmin1, "vfrac_Rmin1p0",
               figsize=(4 * len(M_TURN_VALUES), 5))
print("  Saved: vfrac_Rmin1p0.{pdf,png}")

  Saved: vfrac_Rmin0p5.png
  Saved: vfrac_Rmin1p0.png
  Saved: vfrac_Rmin3p0.png
  Saved: vfrac_Rmin1p0.{pdf,png}
  Saved: vfrac_Rmin1p0.{pdf,png}


In [28]:
# ── CELL 10 : Bubble radius PDFs ─────────────────────────────────────────────
R_bins = np.logspace(-1.5, 1.5, 30)
R_cen  = 0.5 * (R_bins[:-1] + R_bins[1:])

Z_PDF  = [10.0, 9.0, 8.0, 7.5, 7.0, 6.5, 6.0]   # redshifts worth plotting

def plot_pdf_one_z(fig, z):
    axes = fig.subplots(1, len(M_TURN_VALUES), sharey=True)
    for ax, mt in zip(axes, M_TURN_VALUES):
        for fe in F_ESC10_VALUES:
            label = make_label(mt, fe)
            R_arr = bubble_data[label].get(z, np.array([]))
            if len(R_arr) == 0:
                continue
            V_weights      = (4/3) * np.pi * R_arr**3
            vw_hist, _     = np.histogram(R_arr, bins=R_bins,
                                          weights=V_weights)
            vw_hist       /= (vw_hist.sum() + 1e-30)
            xhi            = coeval_data[label][z]['xH_mean']
            ax.step(R_cen, vw_hist, where='mid',
                    color=fe_colors[fe], lw=1.6,
                    label=f'FE={fe:+.1f}')

        ax.set_xscale('log')
        ax.set_xlabel(r'$R_{\rm bubble}$ [pMpc]')
        ax.set_xlim(R_bins[0], R_bins[-1])
        ax.set_title(rf'$M_{{\rm turn}}=10^{{{mt:.1f}}}$', fontsize=10)

        if ax == axes[0]:
            ax.set_ylabel(r'Volume-weighted PDF')
        if ax == axes[-1]:
            ax.legend(fontsize=8, title=r'$\log f_{\rm esc}$',
                      loc='upper left')

    sm = plt.cm.ScalarMappable(cmap='cool',
                                norm=plt.Normalize(min(F_ESC10_VALUES),
                                                   max(F_ESC10_VALUES)))
    sm.set_array([])
    fig.colorbar(sm, ax=axes[-1],
                 label=r'$\log_{10} f_{\rm esc,10}$', shrink=0.8)

# ── Save one PNG per redshift ─────────────────────────────────────────────────
for z in Z_PDF:
    with mpl.rc_context(PNG_STYLE):
        fig = plt.figure(figsize=(4 * len(M_TURN_VALUES), 5),
                         constrained_layout=True)
        plot_pdf_one_z(fig, z)
        fig.suptitle(rf'Bubble PDF at $z={z:.1f}$', fontweight='bold')
        zstr  = f"{z:.1f}".replace('.', 'p')
        fname = f"bubble_pdf_z{zstr}.png"
        fig.savefig(os.path.join(PLOT_DIR, fname),
                    bbox_inches='tight', dpi=150)
        plt.close(fig)
        print(f"  Saved: {fname}")

# ── PDF version for z=7 and z=8 via save_fig_multi ───────────────────────────
for z in [7.0, 8.0]:
    def _plot(fig, _z=z):
        plot_pdf_one_z(fig, _z)
    zstr = f"{z:.1f}".replace('.', 'p')
    save_fig_multi(_plot, f"bubble_pdf_z{zstr}",
                   figsize=(4 * len(M_TURN_VALUES), 5))
    print(f"  Saved: bubble_pdf_z{zstr}.{{pdf,png}}")

# ── Diagnostic: z=7 vs z=8 for fiducial M_TURN only ─────────────────────────
with mpl.rc_context(PNG_STYLE):
    mt_mid = 8.7
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
    for ax, z in zip(axes, [7.0, 8.0]):
        for fe in F_ESC10_VALUES:
            label = make_label(mt_mid, fe)
            R_arr = bubble_data[label].get(z, np.array([]))
            if len(R_arr) == 0:
                continue
            V_weights  = (4/3) * np.pi * R_arr**3
            vw_hist, _ = np.histogram(R_arr, bins=R_bins, weights=V_weights)
            vw_hist   /= (vw_hist.sum() + 1e-30)
            xhi        = coeval_data[label][z]['xH_mean']
            ax.step(R_cen, vw_hist, where='mid',
                    color=fe_colors[fe], lw=1.6,
                    label=rf'FE={fe:+.1f}  $\langle x_{{\rm HI}}\rangle={xhi:.2f}$')
        ax.set_xscale('log')
        ax.set_xlabel(r'$R_{\rm bubble}$ [pMpc]')
        ax.set_title(rf'$z={z:.0f}$, $M_{{\rm turn}}=10^{{{mt_mid}}}$',
                     fontweight='bold')
        ax.set_xlim(R_bins[0], R_bins[-1])
        ax.legend(fontsize=9)

    axes[0].set_ylabel(r'Volume-weighted PDF')
    fig.tight_layout()
    fig.savefig(os.path.join(PLOT_DIR, "diag_bubble_pdf_z7_z8.png"),
                bbox_inches='tight', dpi=150)
    plt.close(fig)
    print("  Saved: diag_bubble_pdf_z7_z8.png")

  Saved: bubble_pdf_z10p0.png
  Saved: bubble_pdf_z9p0.png
  Saved: bubble_pdf_z8p0.png
  Saved: bubble_pdf_z7p5.png
  Saved: bubble_pdf_z7p0.png
  Saved: bubble_pdf_z6p5.png
  Saved: bubble_pdf_z6p0.png
  Saved: bubble_pdf_z7p0.{pdf,png}
  Saved: bubble_pdf_z7p0.{pdf,png}
  Saved: bubble_pdf_z8p0.{pdf,png}
  Saved: bubble_pdf_z8p0.{pdf,png}
  Saved: diag_bubble_pdf_z7_z8.png


In [ ]:
# # ── CELL 11 : 2D slices at fixed redshifts ───────────────────────────────────
# # Grid: models (rows) × redshifts (cols), showing xH_box mid-plane slices.

# def plot_slice_grid(fig):
#     n_models  = len(PARAM_SETS)
#     n_z       = len(SNAP_REDSHIFTS)
#     axes = fig.subplots(n_models, n_z,
#                         gridspec_kw=dict(wspace=0.05, hspace=0.05))

#     for i, (label, lc) in enumerate(lightcones.items()):
#         for j, z in enumerate(SNAP_REDSHIFTS):
#             ax = axes[i, j]
#             box = get_coeval_box(lc, z, 'xH_box')
#             N   = USER_PARAMS.HII_DIM
#             sl  = box[:N, N//2, :N]   # mid-y slice
#             im  = ax.imshow(np.log10(sl + 1e-6),
#                             origin='lower', cmap='RdBu_r',
#                             vmin=-4, vmax=0,
#                             extent=[0, USER_PARAMS.BOX_LEN] * 2)
#             ax.set_xticks([]); ax.set_yticks([])
#             if i == 0:
#                 ax.set_title(rf'$z={z:.0f}$', fontsize=11)
#             if j == 0:
#                 ax.set_ylabel(label, fontsize=9, rotation=90,
#                               va='center', labelpad=4)

#     # Shared colorbar
#     cbar_ax = fig.add_axes([0.92, 0.12, 0.012, 0.76])
#     fig.colorbar(im, cax=cbar_ax, label=r'$\log_{10}(x_{\rm HI})$')

# save_fig_multi(plot_slice_grid, "slice_grid",
#                title=r"$x_{\rm HI}$ slices: models × redshifts",
#                figsize=(3.5 * len(SNAP_REDSHIFTS),
#                         2.8 * len(PARAM_SETS)))


In [29]:
# ── CELL 12 : Fraction of ionised volume in large bubbles ────────────────────
# Asthana+2024 Fig. 10b analogue (volume proxy).

def compute_vion_frac(label, R_min):
    frac_list = []
    for z in z_sorted:
        R_arr = bubble_data[label].get(z, np.array([]))
        if len(R_arr) == 0:
            frac_list.append(0.0)
            continue
        V_large = np.sum((4/3)*np.pi*R_arr[R_arr >= R_min]**3)
        V_tot   = np.sum((4/3)*np.pi*R_arr**3)
        frac_list.append(V_large / V_tot if V_tot > 0 else 0.0)
    return np.array(frac_list)


def plot_vion_panels(fig, R_min):
    axes = fig.subplots(1, len(M_TURN_VALUES), sharey=True)
    for ax, mt in zip(axes, M_TURN_VALUES):
        for fe in F_ESC10_VALUES:
            label = make_label(mt, fe)
            frac  = compute_vion_frac(label, R_min)
            ax.plot(z_sorted, frac, color=fe_colors[fe],
                    lw=1.8, marker='o', ms=3,
                    label=f'FE={fe:+.1f}')

        ax.set_xlabel(r'Redshift $z$')
        ax.set_title(rf'$M_{{\rm turn}}=10^{{{mt:.1f}}}$', fontsize=10)
        ax.set_xlim(min(z_sorted)-0.3, max(z_sorted)+0.3)
        ax.set_ylim(-0.02, 1.05)
        ax.invert_xaxis()
        ax.axhline(1.0, ls='--', color='gray', lw=0.8, alpha=0.5)

        if ax == axes[0]:
            ax.set_ylabel(r'$V_{\rm large} / V_{\rm ion}$')
        if ax == axes[-1]:
            ax.legend(fontsize=8, title=r'$\log f_{\rm esc}$',
                      loc='lower right')

    sm = plt.cm.ScalarMappable(cmap='cool',
                                norm=plt.Normalize(min(F_ESC10_VALUES),
                                                   max(F_ESC10_VALUES)))
    sm.set_array([])
    fig.colorbar(sm, ax=axes[-1],
                 label=r'$\log_{10} f_{\rm esc,10}$', shrink=0.8)

    fig.text(0.01, 0.01, 'Asthana+24 Fig.10b analogue (volume proxy)',
             fontsize=8, alpha=0.5)


# ── One PNG per R_min ─────────────────────────────────────────────────────────
for R_min in [0.5, 1.0, 3.0]:
    with mpl.rc_context(PNG_STYLE):
        fig = plt.figure(figsize=(4 * len(M_TURN_VALUES), 5),
                         constrained_layout=True)
        plot_vion_panels(fig, R_min)
        fig.suptitle(rf'$V_{{\rm large}}(R>{R_min}\ \rm pMpc) / V_{{\rm ion}}$',
                     fontweight='bold')
        Rstr  = str(R_min).replace('.', 'p')
        fname = f"vion_frac_Rmin{Rstr}.png"
        fig.savefig(os.path.join(PLOT_DIR, fname),
                    bbox_inches='tight', dpi=150)
        plt.close(fig)
        print(f"  Saved: {fname}")

# ── PDF for R_min=1 pMpc via save_fig_multi ───────────────────────────────────
def _plot_vion_1(fig):
    plot_vion_panels(fig, 1.0)

save_fig_multi(_plot_vion_1, "vion_frac_Rmin1p0",
               figsize=(4 * len(M_TURN_VALUES), 5))
print("  Saved: vion_frac_Rmin1p0.{pdf,png}")

# ── Diagnostic: mid M_TURN model, all R_min overlaid ─────────────────────────
with mpl.rc_context(PNG_STYLE):
    mt_mid = 8.7
    fig, axes = plt.subplots(1, len(F_ESC10_VALUES),
                             figsize=(4 * len(F_ESC10_VALUES), 5),
                             sharey=True)
    from matplotlib.lines import Line2D

    for ax, fe in zip(axes, F_ESC10_VALUES):
        label = make_label(mt_mid, fe)
        for R_min, ls in zip([0.5, 1.0, 3.0], [':', '-', '--']):
            frac = compute_vion_frac(label, R_min)
            ax.plot(z_sorted, frac, color=fe_colors[fe],
                    ls=ls, lw=1.8, marker='o', ms=3)

        ax.set_xlabel(r'Redshift $z$')
        ax.set_title(rf'FE={fe:+.1f}', fontsize=10)
        ax.set_xlim(min(z_sorted)-0.3, max(z_sorted)+0.3)
        ax.set_ylim(-0.02, 1.05)
        ax.invert_xaxis()
        ax.axhline(1.0, ls='--', color='gray', lw=0.8, alpha=0.4)

    axes[0].set_ylabel(r'$V_{\rm large} / V_{\rm ion}$')

    lh = [Line2D([0],[0], color='gray', ls=ls, lw=1.5,
                 label=rf'$R_{{\min}}={r}$ pMpc')
          for r, ls in zip([0.5, 1.0, 3.0], [':', '-', '--'])]
    axes[-1].legend(handles=lh, fontsize=9)

    fig.suptitle(rf'$M_{{\rm turn}}=10^{{{mt_mid}}}$ — all $R_{{\rm min}}$',
                 fontweight='bold')
    fig.tight_layout()
    fig.savefig(os.path.join(PLOT_DIR, "diag_vion_frac_midMT.png"),
                bbox_inches='tight', dpi=150)
    plt.close(fig)
    print("  Saved: diag_vion_frac_midMT.png")

  Saved: vion_frac_Rmin0p5.png
  Saved: vion_frac_Rmin1p0.png
  Saved: vion_frac_Rmin3p0.png
  Saved: vion_frac_Rmin1p0.{pdf,png}
  Saved: vion_frac_Rmin1p0.{pdf,png}
  Saved: diag_vion_frac_midMT.png
